# 2. Data Understanding and Description

Descriptive tables of the study dataset built by the pipeline (`data/processed/`): flights from `flights_clean.parquet` joined by `row_id` with realized weather at the scheduled departure (origin, `org_`) and arrival (destination, `dst_`) hours from `wx_oracle.parquet`.

All statistics use the **training period (2018–2021)** unless stated otherwise (Table 1 and Table 3 cover the full period). Tables are written to `EDA/data/paper_tables/`.

## 2.1 Load data

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option('display.float_format', lambda x: f'{x:,.2f}')
pd.set_option('display.max_columns', None)

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'README.md').exists())  # project root
DATA = ROOT / 'data' / 'processed'          # outputs of 02_weather_processing and 03_flights_information_settings
TRAIN_END = '2022-01-01'

FLIGHT_COLS = ['row_id', 'FlightDate', 'Airline', 'Origin', 'Dest', 'OriginState', 'DestState',
               'CRSDepHour', 'Month', 'DayOfWeek', 'Distance', 'CRSElapsedTime',
               'ArrDelay', 'ArrDelayMinutes', 'ArrDel15']
WX = ['temp_f', 'dewpt_f', 'humidity', 'wind_mph', 'gust_mph', 'visibility_mi', 'precip_in', 'pressure',
      'ceiling_ft', 'has_rain', 'has_snow', 'has_fog', 'has_mist', 'has_thunder', 'has_freezing',
      'is_heavy', 'has_gust', 'flight_cat_ord']
WX_COLS = ['row_id'] + [f'{p}_{c}' for p in ('org', 'dst') for c in WX]

fl = pd.read_parquet(DATA / 'flights_clean.parquet', columns=FLIGHT_COLS)
wx = pd.read_parquet(DATA / 'wx_oracle.parquet', columns=WX_COLS)   # realized weather at scheduled times (descriptive use)
assert (fl['row_id'].values == wx['row_id'].values).all()
df_all = pd.concat([fl, wx.drop(columns='row_id')], axis=1)
del fl, wx
df_all['FlightDate'] = pd.to_datetime(df_all['FlightDate'])
FLIGHT_CAT = {0: 'VFR', 1: 'MVFR', 2: 'IFR', 3: 'LIFR'}            # FAA category (ceiling and visibility)
for p in ('org', 'dst'):
    df_all[f'{p}_ceiling_cat'] = df_all[f'{p}_flight_cat_ord'].map(FLIGHT_CAT)

df = df_all[df_all['FlightDate'] < TRAIN_END].reset_index(drop=True)   # training period
print(f'All flights: {len(df_all):,} | training period (2018-2021): {len(df):,}')

TABLES_DIR = ROOT / 'EDA' / 'data' / 'paper_tables'
TABLES_DIR.mkdir(parents=True, exist_ok=True)

All flights: 9,981,331 | training period (2018-2021): 8,550,324


In [2]:
# Preview
df.head(3)

,row_id,FlightDate,Airline,Origin,Dest,OriginState,DestState,CRSDepHour,Month,DayOfWeek,Distance,CRSElapsedTime,ArrDelay,ArrDelayMinutes,ArrDel15,org_temp_f,org_dewpt_f,org_humidity,org_wind_mph,org_gust_mph,org_visibility_mi,org_precip_in,org_pressure,org_ceiling_ft,org_has_rain,org_has_snow,org_has_fog,org_has_mist,org_has_thunder,org_has_freezing,org_is_heavy,org_has_gust,org_flight_cat_ord,dst_temp_f,dst_dewpt_f,dst_humidity,dst_wind_mph,dst_gust_mph,dst_visibility_mi,dst_precip_in,dst_pressure,dst_ceiling_ft,dst_has_rain,dst_has_snow,dst_has_fog,dst_has_mist,dst_has_thunder,dst_has_freezing,dst_is_heavy,dst_has_gust,dst_flight_cat_ord,org_ceiling_cat,dst_ceiling_cat
0,0,2018-01-23,Endeavor Air Inc.,ABY,ATL,GA,GA,12,1,2,145.00,62.00,-8.00,0.00,0.00,64.00,29.00,27.00,5.00,16.00,10.00,0.00,29.85,"25,000.00",0.00,0.00,0.00,0.00,0.00,0.00,0.00,1,0,54.00,33.00,45.00,13.00,24.00,10.00,0.00,28.88,"25,000.00",0.00,0.00,0.00,0.00,0.00,0.00,0.00,0,0,VFR,VFR
1,1,2018-01-24,Endeavor Air Inc.,ABY,ATL,GA,GA,12,1,3,145.00,62.00,-6.00,0.00,0.00,51.00,31.00,46.00,8.00,8.00,10.00,0.00,30.08,"25,000.00",0.00,0.00,0.00,0.00,0.00,0.00,0.00,0,0,51.00,25.00,36.00,18.00,18.00,10.00,0.00,29.12,"25,000.00",0.00,0.00,0.00,0.00,0.00,0.00,0.00,0,0,VFR,VFR
2,2,2018-01-25,Endeavor Air Inc.,ABY,ATL,GA,GA,12,1,4,145.00,62.00,-2.00,0.00,0.00,60.00,26.00,27.00,7.00,7.00,10.00,0.00,30.29,"25,000.00",0.00,0.00,0.00,0.00,0.00,0.00,0.00,0,0,55.00,22.00,28.00,5.00,5.00,10.00,0.00,29.36,"25,000.00",0.00,0.00,0.00,0.00,0.00,0.00,0.00,0,0,VFR,VFR


In [3]:
# Column overview
flight_cols = [c for c in df.columns if not c.startswith(('org_','dst_'))]
org_cols = [c for c in df.columns if c.startswith('org_')]
dst_cols = [c for c in df.columns if c.startswith('dst_')]

print(f'Flight columns: {len(flight_cols)}')
print(f'Weather Origin cols: {len(org_cols)}')
print(f'Weather Dest cols: {len(dst_cols)}')
print(f'Total: {len(df.columns)}')

Flight columns: 15
Weather Origin cols: 19
Weather Dest cols: 19
Total: 53


## 2.2 Dataset overview (full period)

In [4]:
# Full study period (2018 - July 2022)
# Basic info
table1_data = {
    'Metric': [
        'Time period',
        'Number of flights (N)',
        'Number of Origin airports',
        'Number of Dest airports',
        'Number of U.S. states',
        'Weather features per endpoint (models)',
        'Delay rate (ArrDel15 = 1)',
        'Mean arrival delay (min)',
        'Median arrival delay (min)',
        'p75 arrival delay (min)',
        'p90 arrival delay (min)',
        'p99 arrival delay (min)',
        'Max arrival delay (min)',
        'Data source (flights)',
        'Data source (weather)',
    ],
    'Value': [
        f"{df_all['FlightDate'].min().date()} to {df_all['FlightDate'].max().date()}",
        f"{len(df_all):,}",
        f"{df_all['Origin'].nunique()}",
        f"{df_all['Dest'].nunique()}",
        f"{df_all['OriginState'].nunique()}",
        '31',
        f"{df_all['ArrDel15'].mean():.2%}",
        f"{df_all['ArrDelayMinutes'].mean():.2f}",
        f"{df_all['ArrDelayMinutes'].median():.2f}",
        f"{df_all['ArrDelayMinutes'].quantile(0.75):.2f}",
        f"{df_all['ArrDelayMinutes'].quantile(0.90):.2f}",
        f"{df_all['ArrDelayMinutes'].quantile(0.99):.2f}",
        f"{df_all['ArrDelayMinutes'].max():.0f}",
        'BTS On-Time Performance',
        'NOAA Local Climatological Data',
    ]
}
table1 = pd.DataFrame(table1_data)
table1.to_csv(TABLES_DIR / 'table1_dataset_overview.csv', index=False)
table1

,Metric,Value
0,Time period,2018-01-01 to 2022-07-31
1,Number of flights (N),"9,981,331"
2,Number of Origin airports,137
3,Number of Dest airports,137
4,Number of U.S. states,10
5,Weather features per endpoint (models),31
6,Delay rate (ArrDel15 = 1),18.05%
7,Mean arrival delay (min),12.92
8,Median arrival delay (min),0.00
9,p75 arrival delay (min),6.00


## 2.3 Breakdown by origin state

In [5]:
# Per-state stats
table2 = df.groupby('OriginState').agg(
    n_flights=('ArrDelayMinutes', 'size'),
    mean_delay=('ArrDelayMinutes', 'mean'),
    median_delay=('ArrDelayMinutes', 'median'),
    p90_delay=('ArrDelayMinutes', lambda x: x.quantile(0.90)),
    pct_delayed=('ArrDel15', 'mean'),
    n_airports=('Origin', 'nunique'),
).round(2).sort_values('n_flights', ascending=False)

table2['pct_delayed'] = (table2['pct_delayed'] * 100).round(2).astype(str) + '%'
table2.columns = ['N flights', 'Mean delay (min)', 'Median delay', 'p90 delay', '% delayed', 'N airports']
table2.to_csv(TABLES_DIR / 'table2_per_state.csv')
table2

,N flights,Mean delay (min),Median delay,p90 delay,% delayed,N airports
OriginState,,,,,,
CA,1652902,10.82,0.00,28.00,16.0%,22
TX,1558038,13.03,0.00,35.00,19.0%,26
FL,1123517,12.79,0.00,34.00,17.0%,19
GA,695402,9.68,0.00,24.00,15.0%,7
NY,692266,14.71,0.00,40.00,19.0%,14
IL,665170,14.70,0.00,42.00,21.0%,11
NC,626991,11.85,0.00,31.00,17.0%,9
CO,571514,14.63,0.00,40.00,20.0%,11
VA,541482,13.35,0.00,34.00,17.0%,9


Delay rates differ between states; every state contributes several hundred thousand flights, sufficient for the tests in notebook 4.

## 2.4 Breakdown by year (full period)
The COVID-19 year 2020 shows a sharp drop in traffic and delay rate; 2022 covers January–July only.

In [6]:
df_all['Year'] = df_all['FlightDate'].dt.year

table3 = df_all.groupby('Year').agg(
    n_flights=('ArrDelayMinutes', 'size'),
    mean_delay=('ArrDelayMinutes', 'mean'),
    pct_delayed=('ArrDel15', 'mean'),
    n_airports=('Origin', 'nunique'),
).round(2)

table3['pct_delayed'] = (table3['pct_delayed'] * 100).round(2).astype(str) + '%'
table3['n_flights_str'] = table3['n_flights'].apply(lambda x: f'{x:,}')
table3.to_csv(TABLES_DIR / 'table3_per_year.csv')
table3

,n_flights,mean_delay,pct_delayed,n_airports,n_flights_str
Year,,,,,
2018,1960594,13.79,20.0%,132,"1,960,594"
2019,2777878,14.56,20.0%,133,"2,777,878"
2020,1635896,6.83,10.0%,136,"1,635,896"
2021,2175956,12.73,18.0%,134,"2,175,956"
2022,1431007,15.83,22.0%,134,"1,431,007"


## 2.5 Top-10 airlines

In [7]:
table4 = df.groupby('Airline').agg(
    n_flights=('ArrDelayMinutes', 'size'),
    mean_delay=('ArrDelayMinutes', 'mean'),
    pct_delayed=('ArrDel15', 'mean'),
).round(2).sort_values('n_flights', ascending=False).head(10)

table4['pct_delayed'] = (table4['pct_delayed'] * 100).round(2).astype(str) + '%'
table4['n_flights_str'] = table4['n_flights'].apply(lambda x: f'{x:,}')
table4['pct_of_total'] = (table4['n_flights'] / len(df) * 100).round(2).astype(str) + '%'
table4.to_csv(TABLES_DIR / 'table4_per_airline.csv')
table4

,n_flights,mean_delay,pct_delayed,n_flights_str,pct_of_total
Airline,,,,,
Southwest Airlines Co.,1620812,9.95,18.0%,"1,620,812",18.96%
American Airlines Inc.,1255629,13.32,18.0%,"1,255,629",14.69%
Delta Air Lines Inc.,1020734,9.57,13.0%,"1,020,734",11.94%
United Air Lines Inc.,932779,12.39,17.0%,"932,779",10.91%
SkyWest Airlines Inc.,737280,15.07,18.0%,"737,280",8.62%
JetBlue Airways,353585,18.13,24.0%,"353,585",4.14%
Envoy Air,304703,11.64,17.0%,"304,703",3.56%
Alaska Airlines Inc.,301323,10.09,18.0%,"301,323",3.52%
Comair Inc.,277335,12.00,17.0%,"277,335",3.24%


## 2.6 Delay distribution
The target `ArrDelayMinutes` = max(0, `ArrDelay`) is zero-inflated and heavy-tailed.

In [8]:
y = df["ArrDelayMinutes"]

def in_range(y, lo, hi):
    if hi is None:
        return (y >= lo).sum()
    return ((y >= lo) & (y < hi)).sum()

total = len(y)
buckets = [
    ('On-time or early (≤0 min)', in_range(y, -np.inf, 0.0001)),
    ('Minor delay (0-15 min)', in_range(y, 0.0001, 15)),
    ('Moderate delay (15-60 min)', in_range(y, 15, 60)),
    ('Major delay (60-180 min)', in_range(y, 60, 180)),
    ('Severe delay (180+ min)', in_range(y, 180, None)),
]

table5 = pd.DataFrame(buckets, columns=['Delay bucket', 'N flights'])
table5['% of total'] = (table5['N flights'] / total * 100).round(2).astype(str) + '%'
table5['N flights'] = table5['N flights'].apply(lambda x: f'{x:,}')
table5.to_csv(TABLES_DIR / 'table5_delay_distribution.csv', index=False)
table5

,Delay bucket,N flights,% of total
0,On-time or early (≤0 min),"5,769,358",67.48%
1,Minor delay (0-15 min),"1,291,795",15.11%
2,Moderate delay (15-60 min),"999,511",11.69%
3,Major delay (60-180 min),"401,340",4.69%
4,Severe delay (180+ min),"88,320",1.03%


In [9]:
# Extreme values check
print("Delay distribution stats:")
print(f"  Skewness: {y.skew():.2f}")  # > 1: strongly right-skewed
print(f"  Kurtosis: {y.kurtosis():.2f}")
print(f"\nPercentiles:")
for p in [50, 75, 90, 95, 99, 99.9]:
    print(f"  p{p}: {y.quantile(p/100):.1f} min")
print(f"\n% early arrivals (signed ArrDelay < 0): {(df['ArrDelay'] < 0).mean():.2%}")
print(f"% zero target (ArrDelayMinutes = 0): {(y == 0).mean():.2%}")
print(f"% positive delay: {(y > 0).mean():.2%}")

Delay distribution stats:
  Skewness: 11.45
  Kurtosis: 235.91

Percentiles:
  p50: 0.0 min
  p75: 6.0 min
  p90: 33.0 min
  p95: 67.0 min
  p99: 182.0 min
  p99.9: 549.0 min

% early arrivals (signed ArrDelay < 0): 65.64%
% zero target (ArrDelayMinutes = 0): 67.48%
% positive delay: 32.52%


## 2.7 Weather features (origin)
Continuous variables, weather-type flags (origin and destination) and FAA flight categories. Fog (`FG`) and mist (`BR`) are separate flags.

In [10]:
# Continuous weather features - Origin
continuous_wx = ['temp_f', 'dewpt_f', 'humidity', 'wind_mph', 'gust_mph',
                 'visibility_mi', 'precip_in', 'pressure', 'ceiling_ft']

wx_stats = df[[f'org_{c}' for c in continuous_wx]].describe().T[['count','mean','std','min','max']].round(2)
wx_stats.index = [c.replace('org_', '') for c in wx_stats.index]
wx_stats.to_csv(TABLES_DIR / 'table6a_weather_continuous.csv')
wx_stats

,count,mean,std,min,max
temp_f,"8,549,172.00",64.32,17.15,-33.00,126.00
dewpt_f,"8,548,888.00",49.59,17.61,-32.00,93.00
humidity,"8,548,878.00",63.01,20.61,1.00,100.00
wind_mph,"8,549,623.00",9.04,5.39,0.00,57.00
gust_mph,"8,549,623.00",11.71,8.74,0.00,91.00
visibility_mi,"8,549,678.00",9.28,2.04,0.00,10.00
precip_in,"8,550,324.00",0.00,0.04,0.00,15.73
pressure,"8,531,778.00",29.24,1.41,21.88,30.82
ceiling_ft,"8,531,141.00","17,207.15","10,037.61",0.00,"25,000.00"


In [11]:
# Weather flag frequency
flags = ['has_rain', 'has_snow', 'has_fog', 'has_mist', 'has_thunder', 'has_freezing', 'is_heavy', 'has_gust']

flag_stats = pd.DataFrame({
    'Origin frequency (%)': [df[f'org_{f}'].mean() * 100 for f in flags],
    'Dest frequency (%)': [df[f'dst_{f}'].mean() * 100 for f in flags],
}, index=flags).round(2)

flag_stats.to_csv(TABLES_DIR / 'table6b_weather_flags.csv')
flag_stats

,Origin frequency (%),Dest frequency (%)
has_rain,6.89,6.78
has_snow,0.87,0.85
has_fog,0.89,0.56
has_mist,6.22,5.46
has_thunder,1.09,1.10
has_freezing,0.13,0.11
is_heavy,0.69,0.67
has_gust,13.76,13.80


In [12]:
# Ceiling category distribution
ceiling_dist = df.groupby('org_ceiling_cat', observed=True).agg(
    n_flights=('ArrDelayMinutes', 'size'),
    mean_delay=('ArrDelayMinutes', 'mean'),
    pct_delayed=('ArrDel15', 'mean'),
).round(2)
ceiling_dist['pct_delayed'] = (ceiling_dist['pct_delayed'] * 100).round(2).astype(str) + '%'
ceiling_dist['% of flights'] = (ceiling_dist['n_flights'] / len(df) * 100).round(2).astype(str) + '%'
ceiling_dist['n_flights'] = ceiling_dist['n_flights'].apply(lambda x: f'{x:,}')
ceiling_dist.to_csv(TABLES_DIR / 'table6c_ceiling_category.csv')
ceiling_dist

,n_flights,mean_delay,pct_delayed,% of flights
org_ceiling_cat,,,,
IFR,"429,682",19.10,26.0%,5.03%
LIFR,"209,594",22.78,28.0%,2.45%
MVFR,"1,026,553",13.56,19.0%,12.01%
VFR,"6,866,931",11.53,16.0%,80.31%


## 2.8 Missing values

In [13]:
missing = df.isna().sum()
missing = missing[missing > 0].sort_values(ascending=False)

if len(missing) > 0:
    missing_df = pd.DataFrame({
        'Column': missing.index,
        'N missing': missing.values,
        '% missing': (missing.values / len(df) * 100).round(3)
    })
    print("Columns with missing values:")
    display(missing_df)
else:
    print('No missing values')

print(f"\nOverall missing rate: {df.isna().sum().sum() / (df.shape[0]*df.shape[1]) * 100:.4f}%")

Columns with missing values:


,Column,N missing,% missing
0,dst_ceiling_ft,21938,0.26
1,dst_pressure,21206,0.25
2,dst_ceiling_cat,20857,0.24
3,org_ceiling_ft,19183,0.22
4,org_pressure,18546,0.22
5,org_ceiling_cat,17564,0.20
6,org_humidity,1446,0.02
7,org_dewpt_f,1436,0.02
8,dst_humidity,1394,0.02
9,dst_dewpt_f,1380,0.02



Overall missing rate: 0.0290%
